# Usage

A synthetic catalogue: 20 000 sources over the sky, with a smooth RM pattern plus measurement noise.

In [ ]:
from __future__ import annotations

import astropy.units as u
import numpy as np
from astropy.coordinates import SkyCoord

from structurefunction import (
    AxisPositionErrors,
    SFResult,
    StencilOptions,
    structure_function,
    structure_function_skymap,
)
from structurefunction.fitting import fit_sf
from structurefunction.plotting import plot_sf, plot_skymap

rng = np.random.default_rng(0)
n = 20_000
coords = SkyCoord(
    rng.uniform(0, 360, n) * u.deg,
    np.degrees(np.arcsin(rng.uniform(-1, 1, n))) * u.deg,
    frame="galactic",
)
l_rad = coords.l.radian
b_rad = coords.b.radian
rm_true = 30 * np.sin(4 * l_rad) * np.cos(3 * b_rad) + 10 * np.sin(
    15 * l_rad + 9 * b_rad
)
rm_err = rng.uniform(2, 8, n) * u.rad / u.m**2
rm = rm_true * u.rad / u.m**2 + rng.normal(0, rm_err.value) * u.rad / u.m**2

## Two-point structure function

Values and errors can be `Quantity` (as here) or plain arrays. Coordinates can be in any frame.

In [ ]:
sf2 = structure_function(
    rm, rm_err, coords, bins=12, max_sep=60 * u.deg, patches=64, rng=1
)
plot_sf(sf2);

`bin_tolerance` (default 0.05) lets the tree group pairs whose separations are within that fraction of a bin width of each other. Set it to 0 for an exact result.

In [ ]:
exact = structure_function(
    rm, rm_err, coords, bins=12, max_sep=60 * u.deg, patches=64, bin_tolerance=0, rng=1
)
print(np.round(np.abs(sf2.sf - exact.sf) / exact.sf_err, 3))

## Three-point structure function

`StencilOptions` sets how close to a straight line the three sources must be, and how closely the two distances from the middle source must agree. `binned` mode is fast on large catalogues; `exact` mode finds every triplet within the tolerances.

In [ ]:
sf3 = structure_function(
    rm,
    rm_err,
    coords,
    n_point=3,
    bins=8,
    min_sep=1 * u.deg,
    max_sep=30 * u.deg,
    patches=64,
    stencil=StencilOptions(angle_tol_deg=10.0, sep_tol=0.1, mode="binned"),
    rng=1,
)
plot_sf(sf3);

## Positional errors

Errors can be circular (`CircularPositionError`) or separate in longitude and latitude. `ra_convention` says whether the longitude error already includes the $\\cos b$ factor.

In [ ]:
pos_err = AxisPositionErrors(
    sigma_ra=np.full(n, 3.0) * u.arcsec,
    sigma_dec=np.full(n, 2.0) * u.arcsec,
    ra_convention="on_sky",
)
sf2_pos = structure_function(
    rm,
    rm_err,
    coords,
    bins=12,
    max_sep=60 * u.deg,
    patches=64,
    pos_errors=pos_err,
    rng=1,
)
print(np.round(sf2_pos.sf / sf2.sf, 4))

## Fitting

Fits use the full covariance between bins. `sampler` is `map`, `emcee` or `nautilus` (the last needs the `evidence` extra).

In [ ]:
fit = fit_sf(sf2, model="broken_power_law", sampler="emcee", rng=2)
for name, lo, med, hi in zip(fit.names, fit.lower, fit.median, fit.upper, strict=True):
    print(f"{name}: {med:.2f} (+{hi - med:.2f} / -{med - lo:.2f})")
plot_sf(sf2, fit=fit, rng=3);

## Saving results

Results write to any astropy table format and read back.

In [ ]:
sf2.write("sf2.ecsv", overwrite=True)
back = SFResult.read("sf2.ecsv")
assert np.allclose(back.sf, sf2.sf)
back.to_table()["separation", "sf", "sf_err", "count"]

## Sky maps

`structure_function_skymap` computes a structure function in every HEALPix cell. Sources around each cell centre are weighted by a cosine or Gaussian taper, times any `weights` you pass. With `fit_model`, each fitted parameter becomes a map. The grid follows the frame of `coords`.

By default (`nside="auto"`) the grid is the coarsest that still puts two pixels across the taper's half-power width, from `recommend_nside`. Finer grids only add strongly overlapping cells.

In [ ]:
from structurefunction import recommend_nside

recommend_nside(30 * u.deg, taper="cosine")

In [ ]:
sky = structure_function_skymap(
    rm,
    rm_err,
    coords,
    radius=30 * u.deg,
    taper="cosine",
    bins=6,
    min_sep=1 * u.deg,
    fit_model="power_law",
    rng=4,
)
plot_skymap(sky, "slope")
plot_skymap(sky, "sf", bin_index=3);

In [ ]:
slope_map = sky.map("slope")  # full-sky HEALPix array, NaN where skipped
sky.write("skymap.fits", overwrite=True)
slope_map.shape